In [3]:
import argparse
import os
import sys
import urllib.request
from datetime import date, timedelta

import ee
import pandas as pd

In [4]:
# Configurations

POINTS_ASSET = "projects/ld-slm/assets/crop_stress/sample_points" 
DRIVE_FOLDER = "crop_stress_s1"
DEFAULT_YEARS = list(range(2021, 2026))
ALL_K = list(range(0, 23))
PASS = "ASCENDING"          
KS = list(range(0, 13))         
NOISE_FLOOR_DB = -30
NODATA = -9999
SEED = 42
N_CHUNKS = 40
VARS = ["vv", "vh", "ns1", "orb"]


In [5]:
def window_dates(year, k):
    start = date(year, 1, 1) + timedelta(days=16 * k)
    end = min(start + timedelta(days=16), date(year + 1, 1, 1))
    return start.isoformat(), end.isoformat()


def band_names(ks=None):
    ks = KS if ks is None else ks
    return [f"{v}_k{k:02d}" for k in ks for v in VARS]


def modis_projection():
    return ee.ImageCollection("MODIS/061/MOD13Q1").first().select("NDVI").projection()


def s1_linear(region, year):
    """Quality-masked S1 collection in linear power for the year (+ margin)."""
    ic = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("orbitProperties_pass", PASS))
        .filterBounds(region)
        .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
    )

    def prep(img):
        db = img.select(["VV", "VH"])
        ok = db.select("VV").gt(NOISE_FLOOR_DB).And(db.select("VH").gt(NOISE_FLOOR_DB))
        lin = ee.Image(10).pow(db.divide(10)).updateMask(ok).rename(["vv", "vh"])
        orb = (lin.select("vv").multiply(0)
               .add(ee.Number(img.get("relativeOrbitNumber_start")))
               .rename("orb").toFloat())
        return lin.addBands(orb).copyProperties(img, ["system:time_start"])

    return ic.map(prep)


def window_bands(w, k):
    """Bands for one 16-day window. Always the same 4 bands, even if the window holds no scenes
    (an empty collection's mean() has NO bands, which breaks everything downstream)."""
    names = [f"vv_k{k:02d}", f"vh_k{k:02d}", f"orb_k{k:02d}"]
    nname = f"ns1_k{k:02d}"
    has = w.size().gt(0)
    placeholder = ee.Image.constant([0, 0, 0]).rename(names).selfMask()      # fully masked
    n_placeholder = ee.Image.constant(0).rename(nname).selfMask()
    mean = ee.Image(ee.Algorithms.If(has, w.mean().rename(names), placeholder))
    n = ee.Image(ee.Algorithms.If(has, w.select("vv").count().rename(nname), n_placeholder))
    return mean.addBands(n)


def year_image(ic, year, modis_proj, ks=None):
    ks = KS if ks is None else ks
    native = ee.Projection("EPSG:4326").atScale(10)
    out = None
    for k in ks:
        s, e = window_dates(year, k)
        img = window_bands(ic.filterDate(s, e), k)
        out = img if out is None else out.addBands(img)
    out = out.setDefaultProjection(native)
    agg = out.reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024).reproject(modis_proj)
    return agg.unmask(NODATA)


def load_points(pid_file=None):
    """Sample points, optionally limited to the pids in a CSV (column 'pid') to save quota.
    Handles pid stored as a number OR as text in the asset (a type mismatch matches nothing)."""
    pts = ee.FeatureCollection(POINTS_ASSET)
    if pid_file:
        raw = pd.read_csv(pid_file, dtype=str)["pid"].astype(str).str.strip().tolist()
        sample = pts.first().get("pid").getInfo()
        if isinstance(sample, (int, float)):
            pids = [int(float(x)) for x in raw]
            print(f"pid is numeric in the asset (e.g. {sample}); matching as numbers")
        else:
            pids = raw
            print(f"pid is text in the asset (e.g. {sample!r}); matching as text")
        pts = pts.filter(ee.Filter.inList("pid", pids))
        print(f"Restricting to {len(pids)} pids from {pid_file}")
    n = pts.size().getInfo()
    print(f"Points found in the asset: {n}")
    if n == 0:
        raise ValueError(
            f"No points matched in {POINTS_ASSET}. Check the asset path, that it lives under the project passed "
            "to --project, and that its 'pid' values match the CSV.")
    return pts


def diagnose(year, pid_file=None):
    """Why is the extraction empty? Print what Sentinel-1 actually has over the points."""
    from collections import Counter
    pts = load_points(pid_file)
    region = pts.geometry().bounds()
    base = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(region)
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01"))
    print(f"S1 scenes over the points' bounding box in {year}: {base.size().getInfo()}")
    for prop in ["instrumentMode", "orbitProperties_pass", "platform_number"]:
        print(f"  {prop}:", base.aggregate_histogram(prop).getInfo())
    pol = base.aggregate_array("transmitterReceiverPolarisation").getInfo()
    print("  polarisation combos:", dict(Counter(tuple(p) for p in pol)))
    iw = base.filter(ee.Filter.eq("instrumentMode", "IW"))
    for ps in ["ASCENDING", "DESCENDING"]:
        c = (iw.filter(ee.Filter.eq("orbitProperties_pass", ps))
             .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
             .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
        print(f"  {ps}: IW VV+VH scenes = {c.size().getInfo()}  | relative orbits:",
              c.aggregate_histogram("relativeOrbitNumber_start").getInfo())


def diagnose_extraction(year, pid_file=None):
    """What does the exact extraction collection look like? (scenes, bands, empty windows)"""
    pts = load_points(pid_file)
    region = pts.geometry().bounds()
    ic = s1_linear(region, year)
    n_ic = ic.size().getInfo()
    print(f"Pass = {PASS}; scenes after the extraction filters in {year}: {n_ic}")
    if n_ic == 0:
        print("  No scenes: nothing to inspect further.")
        return
    print("  band names of the first scene:", ic.first().bandNames().getInfo())
    print("  windows (k, start, scenes):")
    for k in KS:
        s, e = window_dates(year, k)
        print("   ", k, s, ic.filterDate(s, e).size().getInfo())
    yi = year_image(ic, year, modis_projection())
    print("  bands in the year image:", yi.bandNames().size().getInfo(), "(expected", 4 * len(KS), ")")


def add_lonlat(f):
    xy = f.geometry().coordinates()
    return f.set({"lon": xy.get(0), "lat": xy.get(1)})


def export_drive(years, modis_proj, pid_file=None):
    points = load_points(pid_file).map(add_lonlat)
    region = points.geometry().bounds()
    for y in years:
        img = year_image(s1_linear(region, y), y, modis_proj)
        table = img.sampleRegions(
            collection=points, projection=modis_proj, tileScale=16, geometries=False
        ).map(lambda f, y=y: f.set("year", y))
        task = ee.batch.Export.table.toDrive(
            collection=table, description=f"s1_{y}", folder=DRIVE_FOLDER, fileFormat="CSV"
        )
        task.start()
        print(f"Started Drive export for {y} (task id: {task.id})")


def k_groups(ks, size):
    return [ks[i:i + size] for i in range(0, len(ks), size)]


def download_local(years, modis_proj, out_dir, n_chunks=N_CHUNKS, pid_file=None, k_group=4):
    os.makedirs(out_dir, exist_ok=True)
    points = load_points(pid_file).randomColumn("rnd", SEED).map(add_lonlat)
    region = points.geometry().bounds()
    for y in years:
        final = os.path.join(out_dir, f"s1_{y}.csv")
        if os.path.exists(final):
            print(f"{y}: already done, skipping")
            continue
        ic = s1_linear(region, y)
        all_parts, group_dfs = [], []
        for gi, kg in enumerate(k_groups(KS, k_group)):
            img = year_image(ic, y, modis_proj, ks=kg)
            cols = ["pid", "year", "lon", "lat"] + band_names(kg)
            parts = []
            for i in range(n_chunks):
                part = os.path.join(out_dir, f"_part_{y}_g{gi}_{i}.csv")
                if not os.path.exists(part):
                    chunk = points.filter(
                        ee.Filter.And(ee.Filter.gte("rnd", i / n_chunks), ee.Filter.lt("rnd", (i + 1) / n_chunks))
                    )
                    tbl = img.sampleRegions(
                        collection=chunk, projection=modis_proj, tileScale=16, geometries=False
                    ).map(lambda f, y=y: f.set("year", y))
                    url = tbl.getDownloadURL(filetype="csv", selectors=cols)
                    urllib.request.urlretrieve(url, part)
                parts.append(part)
                print(f"  {y}: windows {kg[0]}-{kg[-1]}, chunk {i + 1}/{n_chunks}")
            all_parts += parts
            g = pd.concat([pd.read_csv(p, dtype={"pid": str}) for p in parts], ignore_index=True)
            group_dfs.append(g.drop_duplicates("pid").set_index("pid"))
        df = group_dfs[0]
        for g in group_dfs[1:]:
            df = df.join(g.drop(columns=["year", "lon", "lat"]), how="outer")
        df = df.reset_index().replace(NODATA, float("nan"))
        df.to_csv(final, index=False)
        for p in all_parts:
            os.remove(p)
        print(f"{y}: saved {len(df)} rows x {df.shape[1]} columns -> {final}")


def main(argv=None):
    global PASS, KS
    ap = argparse.ArgumentParser()
    ap.add_argument("--project", required=True)
    ap.add_argument("--years", nargs="*", type=int, default=DEFAULT_YEARS)
    ap.add_argument("--local-dir", default=None, help="Download CSVs here instead of Google Drive")
    ap.add_argument("--chunks", type=int, default=N_CHUNKS)
    ap.add_argument("--pass", dest="pass_dir", default=PASS, choices=["ASCENDING", "DESCENDING"],
                    help="orbit pass to use (use --diagnose to see which one covers your area)")
    ap.add_argument("--k-group", type=int, default=4,
                    help="windows fetched per request")
    ap.add_argument("--kmax", type=int, default=12, help="last 16-day window to extract (0-based; 12 = DOY ~208)")
    ap.add_argument("--diagnose", action="store_true", help="print what Sentinel-1 holds over the points and stop")
    ap.add_argument("--pid-file", default=None, help="CSV with a 'pid' column: only extract these points (saves quota)")
    a = ap.parse_args(argv)
    try:
        ee.Initialize(project=a.project)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=a.project)
    PASS = a.pass_dir
    KS = list(range(0, a.kmax + 1))
    if a.diagnose:
        diagnose(a.years[0], a.pid_file)
        diagnose_extraction(a.years[0], a.pid_file)
        return
    modis_proj = modis_projection()
    if a.local_dir:
        download_local(a.years, modis_proj, a.local_dir, a.chunks, a.pid_file, a.k_group)
    else:
        export_drive(a.years, modis_proj, a.pid_file)


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [4]:
main(["--project", "ld-slm", "--diagnose", "--years", "2026",
      "--pid-file", r"crop_stress\keep_pids.csv"])

pid is text in the asset (e.g. '0'); matching as text
Restricting to 3185 pids from crop_stress\keep_pids.csv
Points found in the asset: 3185
S1 scenes over the points' bounding box in 2026: 239
  instrumentMode: {'IW': 239}
  orbitProperties_pass: {'ASCENDING': 141, 'DESCENDING': 98}
  platform_number: {'A': 102, 'C': 60, 'D': 77}
  polarisation combos: {('VV', 'VH'): 239}
  ASCENDING: IW VV+VH scenes = 141  | relative orbits: {'1.0': 66, '103.0': 75}
  DESCENDING: IW VV+VH scenes = 98  | relative orbits: {'168.0': 36, '95.0': 62}
pid is text in the asset (e.g. '0'); matching as text
Restricting to 3185 pids from crop_stress\keep_pids.csv
Points found in the asset: 3185
Pass = ASCENDING; scenes after the extraction filters in 2026: 141
  band names of the first scene: ['vv', 'vh', 'orb']
  windows (k, start, scenes):
    0 2026-01-01 6
    1 2026-01-17 6
    2 2026-02-02 9
    3 2026-02-18 6
    4 2026-03-06 9
    5 2026-03-22 9
    6 2026-04-07 6
    7 2026-04-23 9
    8 2026-05-09 9